# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bsiddan25/program/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
%pip -q install duckdb huggingface_hub

In [2]:
import os
import getpass

HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

HF_TOKEN = HF_TOKEN or getpass.getpass(
    "Paste your Hugging Face READ token (hf_...): "
)

Paste your Hugging Face READ token (hf_...): ··········


In [3]:
import duckdb
import pandas as pd
import numpy as np

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "dim_content": (
        f"read_parquet('{REL}/dim_content.parquet')"
    ),
    "fact_daily": (
        f"read_parquet("
        f"'{REL}/fact_content_daily_performance/**/*.parquet'"
        f")"
    ),
}

print("DuckDB connection and table paths are ready.")

DuckDB connection and table paths are ready.


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

My feature window is Feb 1st - Apr 30th, 2026. The Outcome window is May 2026.
I want to prioritize pages that were previously visible in April (shown in google seach results and received impressions), not updated recently, and lost impressions from March to April ultimately had fewer impressions.



In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

page_windows = con.sql(f"""
    WITH monthly AS (
        SELECT
            client_hash_id,
            content_hash_id,

            COUNT(DISTINCT CASE
                WHEN report_date BETWEEN DATE '2026-02-01'
                                     AND DATE '2026-02-28'
                 AND gsc_data_available IS TRUE
                THEN report_date
            END) AS feb_days,

            COUNT(DISTINCT CASE
                WHEN report_date BETWEEN DATE '2026-03-01'
                                     AND DATE '2026-03-31'
                 AND gsc_data_available IS TRUE
                THEN report_date
            END) AS mar_days,

            COUNT(DISTINCT CASE
                WHEN report_date BETWEEN DATE '2026-04-01'
                                     AND DATE '2026-04-30'
                 AND gsc_data_available IS TRUE
                THEN report_date
            END) AS apr_days,

            COUNT(DISTINCT CASE
                WHEN report_date BETWEEN DATE '2026-05-01'
                                     AND DATE '2026-05-31'
                 AND gsc_data_available IS TRUE
                THEN report_date
            END) AS may_days,

            SUM(CASE
                WHEN report_date BETWEEN DATE '2026-02-01'
                                     AND DATE '2026-02-28'
                 AND gsc_data_available IS TRUE
                THEN gsc_impressions ELSE 0
            END) AS feb_impressions,

            SUM(CASE
                WHEN report_date BETWEEN DATE '2026-03-01'
                                     AND DATE '2026-03-31'
                 AND gsc_data_available IS TRUE
                THEN gsc_impressions ELSE 0
            END) AS mar_impressions,

            SUM(CASE
                WHEN report_date BETWEEN DATE '2026-04-01'
                                     AND DATE '2026-04-30'
                 AND gsc_data_available IS TRUE
                THEN gsc_impressions ELSE 0
            END) AS apr_impressions,

            SUM(CASE
                WHEN report_date BETWEEN DATE '2026-05-01'
                                     AND DATE '2026-05-31'
                 AND gsc_data_available IS TRUE
                THEN gsc_impressions ELSE 0
            END) AS may_impressions

        FROM {TABLES["fact_daily"]}
        WHERE report_date BETWEEN DATE '2026-02-01'
                              AND DATE '2026-05-31'
        GROUP BY 1, 2
    )

    SELECT
        m.*,
CASE
    WHEN d.content_updated_date <= DATE '2026-04-30'
    THEN DATE_DIFF(
        'day',
        d.content_updated_date,
        DATE '2026-04-30'
    )
    ELSE NULL
END AS days_since_last_update,

CASE
    WHEN d.content_updated_date > DATE '2026-04-30'
    THEN TRUE
    ELSE FALSE
END AS updated_after_feature_window,


        d.content_type,
        d.word_count,


    FROM monthly AS m
    INNER JOIN {TABLES["dim_content"]} AS d
        ON m.client_hash_id = d.client_hash_id
       AND m.content_hash_id = d.content_hash_id

    WHERE m.feb_days >= 20
      AND m.mar_days >= 20
      AND m.apr_days >= 20
      AND m.may_days >= 20
      AND m.mar_impressions > 0
      AND m.apr_impressions > 0

""").df()

print(f"Eligible active page histories: {len(page_windows):,}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Eligible active page histories: 62,558


In [8]:
validation = {
    "rows": len(page_windows),
    "duplicate_pages": page_windows.duplicated(
        ["client_hash_id", "content_hash_id"]
    ).sum(),
    "negative_staleness_values": (
        page_windows["days_since_last_update"] < 0
    ).sum(),
    "missing_staleness_values": (
        page_windows["days_since_last_update"].isna()
    ).sum(),
}

validation

{'rows': 62558,
 'duplicate_pages': np.int64(0),
 'negative_staleness_values': np.int64(0),
 'missing_staleness_values': np.int64(46007)}

In [9]:
page_windows["impressions_90d"] = (
    page_windows["feb_impressions"]
    + page_windows["mar_impressions"]
    + page_windows["apr_impressions"]
)

page_windows["recent_change_pct"] = (
    100
    * (
        page_windows["apr_impressions"]
        - page_windows["mar_impressions"]
    )
    / page_windows["mar_impressions"]
)

page_windows["future_change_pct"] = (
    100
    * (
        page_windows["may_impressions"]
        - page_windows["apr_impressions"]
    )
    / page_windows["apr_impressions"]
)

page_windows["future_decline"] = (
    page_windows["may_impressions"]
    < 0.80 * page_windows["apr_impressions"]
).astype(int)

print(
    "Future-decline base rate:",
    f"{100 * page_windows['future_decline'].mean():.2f}%"
)

staleness_data = page_windows.dropna(
    subset=["days_since_last_update"]
).copy()

staleness_data["staleness_bucket"] = pd.cut(
    staleness_data["days_since_last_update"],
    bins=[-np.inf, 90, 180, 365, np.inf],
    labels=[
        "less_than_90",
        "90_to_179",
        "180_to_364",
        "365_or_more",
    ],
    right=False,
)

staleness_table = (
    staleness_data
    .groupby("staleness_bucket", observed=True)
    .agg(
        n=("content_hash_id", "size"),
        future_decline_rate=("future_decline", "mean"),
        median_future_change_pct=("future_change_pct", "median"),
    )
    .reset_index()
)

staleness_table["future_decline_rate"] = (
    100 * staleness_table["future_decline_rate"]
).round(2)

staleness_table["median_future_change_pct"] = (
    staleness_table["median_future_change_pct"].round(2)
)

print(f"Pages with known staleness: {len(staleness_data):,}")
staleness_table

Future-decline base rate: 48.79%
Pages with known staleness: 16,551


,staleness_bucket,n,future_decline_rate,median_future_change_pct
0,less_than_90,16543,49.11,-19.05
1,90_to_179,5,20.00,-4.69
2,180_to_364,3,66.67,-35.46


This tells us that pages updated less than 90 days has a future decline rate of 49.11 %. Pages updated from 90-179 days has a 20 decline rate and from 180 to 364 has 66.67 decline rate. However, this was only for three pages, which is not enough evidence. Thus, it can be determined that staleness cannot be a factor for determining declining pages.

In [10]:
page_windows["momentum_bucket"] = pd.cut(
    page_windows["recent_change_pct"],
    bins=[-np.inf, -50, -20, 0, 20, np.inf],
    labels=[
        "down_50_or_more",
        "down_20_to_50",
        "down_less_than_20",
        "up_0_to_20",
        "up_more_than_20",
    ],
    right=False,
)

momentum_table = (
    page_windows
    .groupby("momentum_bucket", observed=True)
    .agg(
        n=("content_hash_id", "size"),
        future_decline_rate=("future_decline", "mean"),
        median_future_change_pct=("future_change_pct", "median"),
        median_apr_impressions=("apr_impressions", "median"),
    )
    .reset_index()
)

momentum_table["future_decline_rate"] = (
    100 * momentum_table["future_decline_rate"]
).round(2)

momentum_table["median_future_change_pct"] = (
    momentum_table["median_future_change_pct"].round(2)
)

print(f"Pages tested for momentum: {len(page_windows):,}")
momentum_table

Pages tested for momentum: 62,558


,momentum_bucket,n,future_decline_rate,median_future_change_pct,median_apr_impressions
0,down_50_or_more,13280,52.82,-23.53,313.0
1,down_20_to_50,19115,54.31,-24.39,832.0
2,down_less_than_20,10479,44.86,-15.30,1219.0
3,up_0_to_20,7150,39.71,-10.68,1384.5
4,up_more_than_20,12534,44.56,-13.57,1421.0


A March-to-April loss of at least 20% is associated with a higher May decline rate, but it is an imperfect signal and does not establish causation.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.